# Fetch Article Data from Delta Table

This notebook:
1. Connects to the Delta table: `gold_us_prod.content.gld_cross_brand_live`
2. Fetches Architectural Digest articles
3. Transforms the data to match the expected schema
4. Saves to `data/articles.json` for the Python RAG pipeline

**Run this on Databricks cluster: DDA Cluster**

## Setup: Import Libraries

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, regexp_replace, concat_ws, length, avg, min as spark_min, max as spark_max
import json
from datetime import datetime
from collections import Counter

print("✅ Libraries imported successfully")

## 1. Define Query Parameters

In [ ]:
# Configuration
BRAND = 'Architectural Digest'
START_DATE = '2026-01-01'
TABLE_NAME = 'gold_us_prod.content.gld_cross_brand_live'

print(f"Configuration:")
print(f"  Brand: {BRAND}")
print(f"  Start Date: {START_DATE}")
print(f"  Table: {TABLE_NAME}")

## 2. Fetch Data from Delta Table

In [ ]:
# Query the Delta table
query = f"""
SELECT 
    id,
    hed,
    body,
    full_url
FROM {TABLE_NAME}
WHERE brand = '{BRAND}'
    AND body IS NOT NULL
    AND hed IS NOT NULL
    AND full_url IS NOT NULL
    AND published_date >= '{START_DATE}'
"""

print("Fetching data from Delta table...")
print(f"Query:\n{query}")

In [ ]:
# Execute query
df = spark.sql(query)

# Show sample
article_count = df.count()
print(f"\n✅ Fetched {article_count} articles")
df.show(5, truncate=50)

## 3. Check for Duplicates in Source Data

In [ ]:
# Check for duplicate IDs
duplicate_check = df.groupBy("id").count().filter("count > 1")
duplicate_count = duplicate_check.count()

print(f"\n🔍 Duplicate Check:")
print(f"  Total articles: {article_count}")
print(f"  Duplicate IDs: {duplicate_count}")

if duplicate_count > 0:
    print("\n⚠️  WARNING: Duplicates found!")
    duplicate_check.show(10)
else:
    print("  ✅ No duplicates found")

## 4. Transform Data to Match RAG Schema

In [ ]:
# Transform to match expected schema:
# - id: keep as is (convert to string)
# - hed -> title
# - body -> content
# - full_url -> url

transformed_df = df.select(
    col("id").cast("string").alias("id"),
    trim(col("hed")).alias("title"),
    # Clean body text: remove excessive whitespace, newlines
    regexp_replace(trim(col("body")), r'\s+', ' ').alias("content"),
    trim(col("full_url")).alias("url")
)

print("✅ Schema transformation complete")
transformed_df.printSchema()

## 5. Filter Out Empty Values

In [ ]:
# Filter out any rows with null/empty values after transformation
before_count = transformed_df.count()

transformed_df = transformed_df.filter(
    (col("id").isNotNull()) & 
    (col("title") != "") & 
    (col("content") != "") & 
    (col("url") != "")
)

after_count = transformed_df.count()
filtered_out = before_count - after_count

print(f"\n📊 Filtering Results:")
print(f"  Before: {before_count} articles")
print(f"  After: {after_count} articles")
print(f"  Filtered out: {filtered_out} articles")

transformed_df.show(5, truncate=50)

## 6. Deduplicate by ID (Keep First Occurrence)

In [ ]:
# Remove duplicates by ID
before_dedup = transformed_df.count()

# Use DISTINCT to remove duplicates
deduplicated_df = transformed_df.dropDuplicates(["id"])

after_dedup = deduplicated_df.count()
removed_dups = before_dedup - after_dedup

print(f"\n🔧 Deduplication Results:")
print(f"  Before: {before_dedup} articles")
print(f"  After: {after_dedup} articles")
print(f"  Removed duplicates: {removed_dups}")

if removed_dups > 0:
    print(f"  ⚠️  {removed_dups} duplicate articles were removed")
else:
    print(f"  ✅ No duplicates found")

## 7. Convert to JSON Format

In [ ]:
# Collect data to driver
print("Collecting data to driver...")
articles_list = deduplicated_df.collect()

# Convert to list of dictionaries
articles_json = [
    {
        "id": row.id,
        "title": row.title,
        "content": row.content,
        "url": row.url
    }
    for row in articles_list
]

print(f"\n✅ Converted {len(articles_json)} articles to JSON format")

## 8. Validate JSON Data (Python-side Duplicate Check)

In [ ]:
# Final validation: check for duplicates in Python
ids = [a['id'] for a in articles_json]
titles = [a['title'] for a in articles_json]
urls = [a['url'] for a in articles_json]

id_counts = Counter(ids)
title_counts = Counter(titles)
url_counts = Counter(urls)

dup_ids = {k: v for k, v in id_counts.items() if v > 1}
dup_titles = {k: v for k, v in title_counts.items() if v > 1}
dup_urls = {k: v for k, v in url_counts.items() if v > 1}

print(f"\n🔍 Final Validation:")
print(f"  Total articles: {len(articles_json)}")
print(f"  Unique IDs: {len(set(ids))}")
print(f"  Unique titles: {len(set(titles))}")
print(f"  Unique URLs: {len(set(urls))}")

if dup_ids or dup_titles or dup_urls:
    print("\n⚠️  WARNING: Duplicates detected!")
    if dup_ids:
        print(f"  Duplicate IDs: {len(dup_ids)}")
    if dup_titles:
        print(f"  Duplicate titles: {len(dup_titles)}")
    if dup_urls:
        print(f"  Duplicate URLs: {len(dup_urls)}")
else:
    print("  ✅ All validations passed - no duplicates!")

## 9. Preview Sample Article

In [ ]:
# Show sample article
if articles_json:
    print("\n📄 Sample Article:")
    print(json.dumps(articles_json[0], indent=2))
    
    # Show a few titles
    print("\n📚 First 5 Article Titles:")
    for i, article in enumerate(articles_json[:5], 1):
        print(f"  {i}. {article['title'][:80]}...")

## 10. Calculate Content Statistics

In [ ]:
# Calculate statistics using Spark
stats_df = deduplicated_df.select(
    length(col("content")).alias("content_length")
).agg(
    avg("content_length").alias("avg_length"),
    spark_min("content_length").alias("min_length"),
    spark_max("content_length").alias("max_length")
)

print("\n📊 Content Length Statistics:")
stats_df.show()

## 11. Save to DBFS

In [ ]:
# Save to DBFS
dbfs_path = "/FileStore/rag_articles/articles.json"
local_dbfs_path = "/dbfs" + dbfs_path

# Create directory if it doesn't exist
dbutils.fs.mkdirs("/FileStore/rag_articles/")

# Write JSON file
with open(local_dbfs_path, 'w', encoding='utf-8') as f:
    json.dump(articles_json, f, indent=2, ensure_ascii=False)

print(f"✅ Saved to DBFS: {dbfs_path}")
print(f"   Access via: /dbfs{dbfs_path}")

## 12. Summary Report

In [ ]:
# Final summary
print("\n" + "="*60)
print("📋 SUMMARY REPORT")
print("="*60)
print(f"Total articles fetched: {len(articles_json)}")
print(f"Source: {TABLE_NAME}")
print(f"Brand: {BRAND}")
print(f"Date filter: >= {START_DATE}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Output file: {dbfs_path}")
print("="*60)
print("\n✅ Data fetch complete!")

## 13. Download Instructions

To download the file to your local machine:

### Option 1: Using Databricks CLI
```bash
databricks fs cp dbfs:/FileStore/rag_articles/articles.json ./11-Side-Projects/01\ RAG\ For\ Articles/data/articles.json
```

### Option 2: Using the UI
1. Go to **Data** > **DBFS** > **FileStore** > **rag_articles**
2. Click on `articles.json`
3. Download the file
4. Move it to: `11-Side-Projects/01 RAG For Articles/data/articles.json`

### Option 3: Using Python script
See `scripts/download_articles.py`

### Next Steps
1. Download the file using one of the methods above
2. Place it in: `11-Side-Projects/01 RAG For Articles/data/articles.json`
3. Run your Streamlit app: `streamlit run app.py`